In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import math

In [ ]:
FEATURE_COLUMNS = ['beta_propensity', 'proline_fraction', 'net_charge', 'polar_fraction'] # 'AAT', 'TA', 'a3vSA' <- nie ma ich tu bo nie można obliczyć na bieżąco w treningu

#path = "files/df_ml_good_with_features.csv"
path = "df_ml_good_with_features.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()

(1934, 21)


,id,sequence,length,class,merge_id,a3vSA,AAT,Na4vSS,THSA,nHS,...,net_charge,hydrophobicity,polar_fraction,nonpolar_fraction,acidic_fraction,basic_fraction,aromatic_fraction,beta_propensity,seq_entropy,proline_fraction
0,GI_171848907_PDB_2RNM_A__1_1,GRNSAKDIRTEERARVQLGNV,21,1,GI_171848907_PDB_2RNM_A__1_1_0,-0.457,0.190,-0.494,0.190,1.0,...,2.0,-1.185714,0.238095,0.380952,0.142857,0.238095,0.000000,0.953333,3.535175,0.000000
1,GI_342871650_GB_EGU74155_1_1,VRIYAKDIKSEEMARVRVGNE,21,1,GI_342871650_GB_EGU74155_1_1_0,-0.160,2.140,-0.165,1.871,1.0,...,1.0,-0.676190,0.142857,0.428571,0.190476,0.238095,0.047619,0.990000,3.427333,0.000000
2,GI_342887385_GB_EGU86897_1_1,GKNSAGRINGPGMVNIGNS,19,1,GI_342887385_GB_EGU86897_1_1_0,-0.256,1.438,-0.256,1.438,1.0,...,2.0,-0.563158,0.315789,0.578947,0.000000,0.105263,0.000000,0.937368,3.005315,0.052632
3,GI_347837243_EMB_CCD5181_1_1,HRIKIGKVTQASNAKAVIGVH,21,1,GI_347837243_EMB_CCD5181_1_1_0,-0.001,4.054,-0.008,4.054,2.0,...,4.2,-0.019048,0.190476,0.523810,0.000000,0.285714,0.000000,1.081429,3.296148,0.000000
4,GI_475677570_GB_EMT74561_1_1,VRNYASEIKGDEDAKVRLGND,21,1,GI_475677570_GB_EMT74561_1_1_0,-0.436,0.570,-0.435,0.000,0.0,...,-1.0,-1.138095,0.190476,0.380952,0.238095,0.190476,0.047619,0.912381,3.499228,0.000000


In [ ]:
MAX_LEN = 50

FEATURE_COLUMNS = [
    'beta_propensity',
    'proline_fraction',
    'net_charge',
    'polar_fraction'
]

# scaling (IMPORTANT: before dataset)
scaler = StandardScaler()
df[FEATURE_COLUMNS] = scaler.fit_transform(df[FEATURE_COLUMNS])


PAD_IDX = 0

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}
idx_to_aa = {i+1: aa for i, aa in enumerate(AMINO_ACIDS)}

EOS = len(AMINO_ACIDS) + 1
VOCAB_SIZE = EOS + 1


def encode_sequence(seq):
    seq = seq[:MAX_LEN-1]
    tokens = [aa_to_idx.get(a, PAD_IDX) for a in seq]
    tokens.append(EOS)
    tokens += [PAD_IDX] * (MAX_LEN - len(tokens))
    return torch.tensor(tokens)


def decode_sequence(tokens):
    out = []
    for t in tokens:
        if t == EOS:
            break
        if t != PAD_IDX:
            out.append(idx_to_aa[t])
    return "".join(out)


class ProteinDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        seq = encode_sequence(row["sequence"])

        features = torch.tensor(
            row[FEATURE_COLUMNS].astype(float).to_numpy(),
            dtype=torch.float32
        )

        return seq, features

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## Encoder, Decoder, cVAE

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, latent_dim, feature_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.len_proj = nn.Linear(1, 16)

        self.fc_mu = nn.Linear(d_model + 32 + 16, latent_dim)
        self.fc_logvar = nn.Linear(d_model + 32 + 16, latent_dim)

    def forward(self, x, features):
        x_emb = self.embedding(x)
        h = self.encoder(x_emb).mean(dim=1)

        f = self.feature_mlp(features)

        length = (x != PAD_IDX).sum(dim=1, keepdim=True).float()
        len_vec = self.len_proj(length)

        h = torch.cat([h, f, len_vec], dim=1)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar, f, len_vec

In [ ]:
def reparam(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, latent_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        self.latent_to_ctx = nn.Linear(latent_dim, d_model)
        self.len_to_ctx = nn.Linear(16, d_model)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=2)

        self.output = nn.Linear(d_model, vocab_size)

        self.memory_proj = nn.Linear(32, d_model)

    def forward(self, tgt, z, f_enc, len_vec, tgt_mask=None):

        tgt_emb = self.embedding(tgt)

        memory = self.memory_proj(f_enc)

        z_ctx = self.latent_to_ctx(z).unsqueeze(1)
        len_ctx = self.len_to_ctx(len_vec).unsqueeze(1)

        memory = memory + z_ctx + len_ctx

        out = self.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask
        )

        return self.output(out)

In [ ]:
class ProteinCVAE(nn.Module):
    def __init__(self, vocab_size, feature_dim, d_model=128, latent_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)

        # ===== ENCODER =====
        enc_layer = nn.TransformerEncoderLayer(d_model, 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

        self.feature_proj = nn.Linear(feature_dim, d_model)

        self.fc_mu = nn.Linear(d_model, latent_dim)
        self.fc_logvar = nn.Linear(d_model, latent_dim)

        # ===== DECODER =====
        dec_layer = nn.TransformerDecoderLayer(d_model, 4, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=2)

        self.latent_to_memory = nn.Linear(latent_dim, d_model)

        self.output = nn.Linear(d_model, vocab_size)

        self.feature_predictor = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim)
        )

    def encode(self, x, features):
        x = self.embedding(x)
        h = self.encoder(x).mean(dim=1)

        h = h + self.feature_proj(features)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def decode(self, tgt, z):
        tgt = self.embedding(tgt)

        memory = self.latent_to_memory(z).unsqueeze(1)  # [B,1,D]

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1)
        ).to(tgt.device)

        out = self.decoder(tgt, memory, tgt_mask=tgt_mask)

        return self.output(out)

    def forward(self, x, features):
        mu, logvar = self.encode(x, features)

        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)

        inp = x[:, :-1]
        tgt = x[:, 1:]

        logits = self.decode(inp, z)

        pred_feat = self.feature_predictor(z)

        return logits, tgt, mu, logvar, pred_feat

In [ ]:
def loss_fn(logits, target, mu, logvar, feat_pred, feat_true,
            beta=0.1, alpha=1.0):

    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target.reshape(-1),
        ignore_index=PAD_IDX
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    feat_loss = F.mse_loss(feat_pred, feat_true)

    return recon + beta*kl + alpha*feat_loss, recon, kl, feat_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ProteinCVAE(
    vocab_size=VOCAB_SIZE,
    feature_dim=len(FEATURE_COLUMNS)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
print(device)

cuda


In [ ]:
# split danych
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# datasety
train_dataset = ProteinDataset(train_df)
val_dataset = ProteinDataset(val_df)

# loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    drop_last=False
)

In [ ]:
best_val = float("inf")
patience = 5
counter = 0
best_state = None
EPOCHS = 60

best_val_loss = float("inf")
counter = 0


def loss_fn(logits, target, mu, logvar, feat_pred, feat_true,
            beta=0.1, alpha=1.0):

    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target.reshape(-1),
        ignore_index=PAD_IDX
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    feat_loss = F.mse_loss(feat_pred, feat_true)

    total = recon + beta * kl + alpha * feat_loss

    return total, recon, kl, feat_loss


for epoch in range(EPOCHS):

    beta = min(1.0, epoch / 20)

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for x, f in train_loader:
        x, f = x.to(device), f.to(device)

        optimizer.zero_grad()

        logits, tgt, mu, logvar, feat_pred = model(x, f)

        loss, recon, kl, feat = loss_fn(
            logits, tgt, mu, logvar,
            feat_pred, f,
            beta=beta,
            alpha=1.0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, f in val_loader:
            x, f = x.to(device), f.to(device)

            logits, tgt, mu, logvar, feat_pred = model(x, f)

            loss, _, _, _ = loss_fn(
                logits, tgt, mu, logvar,
                feat_pred, f,
                beta=beta,
                alpha=1.0
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1:03d} | train {train_loss:.4f} | val {val_loss:.4f} | beta {beta:.2f}")

    # ================= EARLY STOP =================
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "../best_cvae.pt")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break


# ===== LOAD BEST MODEL =====
model.load_state_dict(torch.load("../best_cvae.pt"))
torch.save(model.state_dict(), "cvae_final.pt")

Epoch 001 | train 3.2496 | val 2.8553 | beta 0.00
Epoch 002 | train 2.8200 | val 2.7447 | beta 0.05
Epoch 003 | train 2.7327 | val 2.6669 | beta 0.10
Epoch 004 | train 2.6710 | val 2.6477 | beta 0.15
Epoch 005 | train 2.6325 | val 2.6114 | beta 0.20
Epoch 006 | train 2.5948 | val 2.6135 | beta 0.25
Epoch 007 | train 2.5680 | val 2.5669 | beta 0.30
Epoch 008 | train 2.5423 | val 2.5983 | beta 0.35
Epoch 009 | train 2.5261 | val 2.5950 | beta 0.40
Epoch 010 | train 2.5066 | val 2.5417 | beta 0.45
Epoch 011 | train 2.4813 | val 2.5550 | beta 0.50
Epoch 012 | train 2.4518 | val 2.5752 | beta 0.55
Epoch 013 | train 2.4347 | val 2.5265 | beta 0.60
Epoch 014 | train 2.4244 | val 2.5352 | beta 0.65
Epoch 015 | train 2.3907 | val 2.5787 | beta 0.70
Epoch 016 | train 2.3762 | val 2.5248 | beta 0.75
Epoch 017 | train 2.3412 | val 2.5810 | beta 0.80
Epoch 018 | train 2.3288 | val 2.5764 | beta 0.85
Epoch 019 | train 2.2995 | val 2.6183 | beta 0.90
Epoch 020 | train 2.2725 | val 2.5947 | beta 0.95


In [ ]:
def generate(model, features, max_len=50, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        features = features.unsqueeze(0).to(device)

        dummy_input = torch.zeros((1, max_len), dtype=torch.long).to(device)

        mu, logvar = model.encode(dummy_input, features)
        z = mu

        seq = torch.zeros((1, 1), dtype=torch.long).to(device)

        for _ in range(max_len):

            logits = model.decode(seq, z)

            probs = F.softmax(logits[:, -1] / temperature, dim=-1)

            next_token = torch.multinomial(probs, 1)

            seq = torch.cat([seq, next_token], dim=1)

            if next_token.item() == EOS:
                break

        return decode_sequence(seq[0].cpu().numpy())

In [ ]:
#best_state = "files/models/cvae_protein_model.pt"
best_state = "best_cvae.pt"
state_dict = torch.load(best_state)
model.load_state_dict(state_dict)

sample_features = scaler.transform(
    df[FEATURE_COLUMNS].iloc[10].values.reshape(1, -1)
)
sample_features = torch.tensor(sample_features[0], dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated:", generated_seq)
# Oryginalna sekwencja z pierwszego wiersza
original_seq = df["sequence"].iloc[10]
print("Original sequence:", original_seq)

Generated: RASKAAKRAKHNECGERERNGKAMD
Original sequence: TKQNIRDVKTTGNSIALAGLI


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
original_seq = df["sequence"].iloc[0]
print("Original sequence:", original_seq)
print("Generated sequence:", generated_seq)

# funkcja do cech (już była wcześniej)
def compute_features_from_sequence(seq):
    aa_list = list(seq)
    L = len(aa_list)

    kd = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,
          'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,
          'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}

    hydrophobicity = sum([kd.get(a,0) for a in aa_list])/L
    pos = sum(a in "KRH" for a in aa_list)
    neg = sum(a in "DE" for a in aa_list)
    charge = pos - neg
    from collections import Counter
    counts = Counter(aa_list)
    probs = torch.tensor([counts.get(a,0)/L for a in AMINO_ACIDS], dtype=torch.float32)
    entropy = -torch.sum(probs * torch.log(probs + 1e-8))
    aromatic_fraction = sum(a in "FWY" for a in aa_list)/L
    proline_fraction = sum(a=="P" for a in aa_list)/L
    pI = (pos + 1) / (neg + 1)
    instability_index = sum(a in "DEKPQR" for a in aa_list)/L

    return {
        "hydrophobicity": hydrophobicity,
        "charge": charge,
        "entropy": entropy.item(),
        "aromatic_fraction": aromatic_fraction,
        "proline_fraction": proline_fraction,
        "pI": pI,
        "instability_index": instability_index
    }

# 🔹 Obliczenie cech
orig_features = compute_features_from_sequence(original_seq)
gen_features = compute_features_from_sequence(generated_seq)

# 🔹 Porównanie w tabeli
import pandas as pd
comparison_df = pd.DataFrame([orig_features, gen_features], index=["Original", "Generated"])
print("\nFeature comparison:")
print(comparison_df)

Original sequence: GRNSAKDIRTEERARVQLGNV
Generated sequence: RASKAAKRAKHNECGERERNGKAMD

Feature comparison:
           hydrophobicity  charge   entropy  aromatic_fraction  \
Original        -1.185714       2  2.450396                0.0   
Generated       -1.840000       5  2.210637                0.0   

           proline_fraction   pI  instability_index  
Original                0.0  1.5           0.428571  
Generated               0.0  2.0           0.480000  


In [ ]:
import re
import torch
import numpy as np

df = pd.read_csv("df_ml_good_with_features.csv")

generated = []

N_ROWS = 1000
N_PER_ROW = 100
MAX_LEN = 50
device = next(model.parameters()).device


def perturb_features(row, noise_level=0.1):
    feats = row[FEATURE_COLUMNS].values.astype(float)

    noise = np.random.normal(0, noise_level, size=len(feats))
    feats = feats + noise

    feats = scaler.transform(feats.reshape(1, -1))[0]
    return torch.tensor(feats, dtype=torch.float32)


def generate_cvae(model, features, max_len=50, temperature=0.8):
    model.eval()

    with torch.no_grad():
        features = features.to(device)

        dummy = torch.zeros((1, max_len), dtype=torch.long).to(device)

        mu, logvar = model.encode(dummy, features.unsqueeze(0))

        # sampling latent space (ważne!)
        z = mu + torch.randn_like(mu) * 0.1

        seq = torch.zeros((1, 1), dtype=torch.long).to(device)

        for _ in range(max_len):

            logits = model.decode(seq, z)

            probs = torch.softmax(logits[:, -1] / temperature, dim=-1)

            token = torch.multinomial(probs, 1)

            seq = torch.cat([seq, token], dim=1)

            if token.item() == EOS:
                break

        return decode_sequence(seq[0].cpu().numpy())


# ================= MAIN LOOP =================

sample_df = df.sample(N_ROWS, random_state=42)

for _, row in sample_df.iterrows():

    for _ in range(N_PER_ROW):

        feats = perturb_features(row, noise_level=0.1)

        seq = generate_cvae(
            model,
            feats,
            max_len=MAX_LEN,
            temperature=0.8
        )

        # ===== CLEANING =====
        seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)

        if len(seq) >= 5:
            generated.append(seq)

In [ ]:
print("Generated:", len(generated))

Generated: 6823


In [ ]:
df_out = pd.DataFrame({
    "sequence": generated,
    "length": [len(s) for s in generated]
})

df_out.to_csv("generated_sequences-cvae.csv", index=False)

print("Generated:", len(generated))
print("Saved to: generated_sequences.csv")

Generated: 6823
Saved to: generated_sequences.csv


In [ ]:
for i in range(100):
    print(f"{i} {generated[i]}")

0 MMGMA
1 IEMKA
2 MPKME
3 EPVMCEK
4 WEMIVA
5 MSWAGKME
6 EMKWP
7 YAMWIE
8 MDAKMMEEILE
9 LMAMMK
10 PLRVW
11 GKWAE
12 WVPMI
13 WALGEP
14 KWAEMPREPMG
15 EKFMKKWA
16 WEPMA
17 EHAMMK
18 KMMME
19 KMPWKE
20 MKMWE
21 MMGMK
22 MGEMK
23 FKWGKE
24 HMKKM
25 RMKMGM
26 MMAEFMMKG
27 KEHWA
28 RPKWKE
29 MGMMKM
30 AMWSEWD
31 KMKEPKGMM
32 KWAEE
33 MKMKE
34 KLEFHK
35 MKCVFE
36 RSKAKMEEG
37 HKMVWAMEPM
38 KMMGAE
39 MKVFE
40 RMCME
41 KWVRE
42 KGKVF
43 RMMVK
44 RMPKMKWDKFM
45 WNMAMLMMKARNE
46 MKGMMR
47 EPRVK
48 MFTIP
49 MNKRR
50 MKHMWA
51 ERGKCLA
52 MVFFK
53 RGMKA
54 WVKPF
55 KSRWVWGM
56 EVKKRD
57 EKWVKM
58 WEEAKWVK
59 AMLMG
60 PMKIE
61 AREKMMN
62 AEKCFP
63 WKPMMV
64 MPMIKKE
65 WPVEKIM
66 NKKWAME
67 TLVRRP
68 TKIYRPQM
69 RPKYM
70 RPRYK
71 KVQRK
72 PVRKKIM
73 KFPVKIR
74 PVKMHR
75 RWKPVKQ
76 MKKVF
77 SRWNK
78 IPKRQVNLF
79 RPVFIKYKK
80 PAIRP
81 QKPKN
82 RPMKK
83 RNSHKFWVR
84 RNYSMR
85 KWKPR
86 RVNKPR
87 PVKKIR
88 NRWKA
89 MKCVFR
90 KIFPR
91 KQVPR
92 KQVRGNK
93 RWPVKK
94 RHMPRQNK
95 PVPRITKYR
96 RLKFP
97 MFKMPR
98